In [1]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

num_items = 100
item_codes = [f"IOL-OPT-{str(i).zfill(3)}" for i in range(1, num_items + 1)]

# ==========================================
# 1. Generate machine_capacities 
# (Defines shop floor bottlenecks, existing WIP, and hourly rates)
# ==========================================
machines = ['CNC_MILL', 'LENS_POLISHER', 'COATING_CHAMBER', 'ASSEMBLY_STATION', 'TESTING_RIG']
available_hours = [120.0, 80.0, 80.0, 160.0, 80.0]
# Existing Work-in-Progress hours already tying up the machines
wip_hours = [45.0, 20.0, 50.0, 30.0, 15.0] 
# Hourly operating cost (INR) for each machine
operating_rates = [1500.0, 800.0, 3000.0, 500.0, 1000.0]

df_machines = pd.DataFrame({
    'machine_id': machines,
    'available_weekly_hours': available_hours,
    'existing_wip_hours': wip_hours,
    'operating_cost_per_hour': operating_rates
})

# Dictionary for quick rate lookups during synthetic data generation
machine_rate_dict = dict(zip(machines, operating_rates))

# ==========================================
# 2. Generate production_routings
# (Defines the exact physics of making each item)
# ==========================================
routing_records = []
for item in item_codes:
    num_ops = np.random.randint(1, 4)
    ops_machines = np.random.choice(machines, num_ops, replace=False)
    
    for seq, machine in enumerate(ops_machines):
        routing_records.append({
            'item_id': item,
            'operation_sequence': (seq + 1) * 10,
            'machine_id': machine,
            'setup_hours': round(np.random.uniform(1.0, 8.0), 1),
            'run_hours_per_unit': round(np.random.uniform(0.1, 2.5), 2)
        })

df_routings = pd.DataFrame(routing_records)

# ==========================================
# 3. Generate mto_item_master
# (Decoupled costs and introduced scrap/yield loss)
# ==========================================
is_critical = np.random.choice([True, False], num_items, p=[0.15, 0.85])
raw_material_costs = np.round(np.random.uniform(200, 25000, num_items), 2)
# Scrap rate between 1% and 8%
scrap_percentages = np.round(np.random.uniform(0.01, 0.08, num_items), 3)

df_items = pd.DataFrame({
    'item_id': item_codes,
    'description': [f"Opto-Electronic Assembly {i}" for i in range(1, num_items + 1)],
    'is_critical_component': is_critical,
    'raw_material_cost_per_unit': raw_material_costs,
    'expected_scrap_percentage': scrap_percentages
})

# ==========================================
# 4. Generate external_supplier_quotes & active_indents
# ==========================================
supplier_prices = []
lead_times = []
contract_values = []

# Calculate the true in-house cost to generate realistic quotes & contracts
for i, item in enumerate(item_codes):
    # Base material cost
    rm_cost = raw_material_costs[i]
    
    # Calculate machine operating costs from routings
    item_routings = df_routings[df_routings['item_id'] == item]
    machine_cost = 0
    for _, row in item_routings.iterrows():
        machine_cost += row['run_hours_per_unit'] * machine_rate_dict[row['machine_id']]
    
    true_variable_cost = rm_cost + machine_cost
    
    # Generate realistic supplier prices based on internal costs
    if is_critical[i]:
        supplier_prices.append(round(true_variable_cost * np.random.uniform(1.5, 3.0), 2))
        lead_times.append(np.random.randint(120, 200))
    else:
        supplier_prices.append(round(true_variable_cost * np.random.uniform(0.6, 1.2), 2))
        lead_times.append(np.random.randint(30, 150))
        
    # MoD contract value (Assume 15% markup on true variable cost)
    contract_values.append(round(true_variable_cost * 1.15, 2))

# QA parameters
qa_hours = np.round(np.random.uniform(0.1, 0.5, num_items), 2)
qa_hourly_rate = 850.0 
qa_costs = np.round(qa_hours * qa_hourly_rate, 2)

# Build Supplier Quotes Table
df_quotes = pd.DataFrame({
    'item_id': item_codes,
    'vendor_price_per_unit': supplier_prices,
    'vendor_lead_time_days': lead_times,
    'vendor_moq': np.random.choice([1, 10, 50, 100], num_items, p=[0.4, 0.3, 0.2, 0.1]),
    'excess_salvage_value_pct': np.random.choice([0.0, 0.25, 0.50], num_items, p=[0.6, 0.3, 0.1]),
    'vendor_reliability_score': np.round(np.random.uniform(0.75, 0.99, num_items), 2),
    'incoming_qa_hours_per_unit': qa_hours,
    'incoming_qa_cost_per_unit': qa_costs
})

# Build Active Indents (Orders) Table
order_quantities = np.random.randint(10, 200, num_items)
delivery_periods = np.random.choice([90, 120, 180], num_items)

df_orders = pd.DataFrame({
    'order_id': [f"MOD-ORD-{str(i).zfill(4)}" for i in range(1001, 1001 + num_items)],
    'item_id': item_codes,
    'order_quantity': order_quantities,
    'delivery_period_days': delivery_periods,
    'ld_rate_per_week': 0.005,  # 0.5% LD per week
    'max_ld_percentage': 0.10,  # 10% maximum LD
    'contract_value_per_unit': contract_values 
})

# ==========================================
# Save all to CSV
# ==========================================
df_items.to_csv("mto_item_master.csv", index=False)
df_machines.to_csv("machine_capacities.csv", index=False)
df_routings.to_csv("production_routings.csv", index=False)
df_quotes.to_csv("external_supplier_quotes.csv", index=False)
df_orders.to_csv("active_indents.csv", index=False)

print("Optimization-ready datasets successfully generated and saved.")

Optimization-ready datasets successfully generated and saved.
